In [5]:
import os
import cv2

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from src.utils import HandDetection, PoseDetection, FaceDetection


# =========================
# Open Camera
# =========================

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    raise RuntimeError("Cannot open camera.")


# =========================
# Detection
# =========================

hand_detection = HandDetection(
    min_hand_detection_confidence=0.2
)

pose_detection = PoseDetection(
    min_pose_detection_confidence=0.2
)

face_detection = FaceDetection()


# =========================
# Camera Info
# =========================

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"width: {width}, height: {height}")
print(f"fps: {fps}")
print("Camera started.")
print("Press 'q' to quit.")


# =========================
# Main Loop
# =========================

frame_index = 0

while True:

    success, frame = cap.read()

    if not success:
        print("Cannot read frame from camera.")
        break

    # BGR -> RGB
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # MediaPipe video timestamp
    timestamp_ms = int(
        frame_index * 1000 / fps
    )

    # =========================
    # Hand Detection
    # =========================

    detection_hand_results = hand_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    # =========================
    # Pose Detection
    # =========================

    detection_pose_results = pose_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    # =========================
    # Face Detection
    # =========================

    detection_face_results = face_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    # =========================
    # Draw Landmarks
    # =========================

    output = hand_detection.draw_landmarks_on_image(
        rgb_frame.copy(),
        detection_hand_results
    )

    output = pose_detection.draw_landmarks_on_image(
        output,
        detection_pose_results
    )

    output = face_detection.draw_lips_on_image(
        output,
        detection_face_results
    )

    # RGB -> BGR
    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    # =========================
    # Display
    # =========================

    cv2.imshow(
        "Camera - Landmarks + Lips",
        output
    )

    frame_index += 1

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# =========================
# Release
# =========================

cap.release()
cv2.destroyAllWindows()

hand_detection.close()
pose_detection.close()
face_detection.close()

print("Camera stopped.")

width: 640, height: 480
fps: 30
Camera started.
Press 'q' to quit.


AttributeError: 'PoseDetection' object has no attribute 'close'